# Listen To One Speaker Across Systems

Choose a speaker from the selected ASVspoof5 subset and listen to that speaker's `bonafide` plus spoof samples across systems.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Audio, display


In [2]:
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
CSV_PATH = PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'analysis_subset_a01_a04_balanced_binary' / 'tcav_analysis_subset_spoof.csv'

df = pd.read_csv(CSV_PATH)
print('CSV_PATH =', CSV_PATH)
print('rows =', len(df))
print('speakers =', df['speaker_id'].nunique())
print('systems =', sorted(df['system_id'].astype(str).unique().tolist()))


CSV_PATH = /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/analysis_subset_a01_a04_balanced_binary/tcav_analysis_subset_spoof.csv
rows = 192
speakers = 6
systems = ['A01', 'A02', 'A03', 'A04', 'bonafide']


In [3]:
speaker_summary = (
    df.groupby(['speaker_id', 'system_id'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
display(speaker_summary)
print('Available speakers:', sorted(df['speaker_id'].unique().tolist()))


system_id,speaker_id,A01,A02,A03,A04,bonafide
0,T_0519,4,4,4,4,16
1,T_0938,4,4,4,4,16
2,T_2070,4,4,4,4,16
3,T_3839,4,4,4,4,16
4,T_4279,4,4,4,4,16
5,T_5164,4,4,4,4,16


Available speakers: ['T_0519', 'T_0938', 'T_2070', 'T_3839', 'T_4279', 'T_5164']


## Choose Speaker
Set `TARGET_SPEAKER` to one of the speaker IDs printed above.


In [4]:
TARGET_SPEAKER = sorted(df['speaker_id'].unique().tolist())[0]
SYSTEM_ORDER = ['bonafide', 'A01', 'A02', 'A03', 'A04']
PICK_INDEX_PER_SYSTEM = 0

speaker_df = df[df['speaker_id'].eq(TARGET_SPEAKER)].copy()
speaker_df['system_id'] = speaker_df['system_id'].astype(str)
speaker_df = speaker_df.sort_values(['system_id', 'utt_id']).reset_index(drop=True)

print('TARGET_SPEAKER =', TARGET_SPEAKER)
display(speaker_df[['speaker_id', 'utt_id', 'label', 'system_id', 'path']].head(20))


TARGET_SPEAKER = T_0519


,speaker_id,utt_id,label,system_id,path
0,T_0519,T_0000002182,spoof,A01,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
1,T_0519,T_0000003512,spoof,A01,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
2,T_0519,T_0000003998,spoof,A01,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
3,T_0519,T_0000004271,spoof,A01,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
4,T_0519,T_0000003194,spoof,A02,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
5,T_0519,T_0000005643,spoof,A02,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
6,T_0519,T_0000007886,spoof,A02,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
7,T_0519,T_0000009889,spoof,A02,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
8,T_0519,T_0000007618,spoof,A03,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
9,T_0519,T_0000010108,spoof,A03,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...


In [5]:
listen_rows = []
for sys_id in SYSTEM_ORDER:
    pool = speaker_df[speaker_df['system_id'].eq(sys_id)].copy().reset_index(drop=True)
    if pool.empty:
        continue
    pick_idx = min(PICK_INDEX_PER_SYSTEM, len(pool) - 1)
    listen_rows.append(pool.iloc[pick_idx])

listen_df = pd.DataFrame(listen_rows).reset_index(drop=True)
display(listen_df[['speaker_id', 'utt_id', 'label', 'system_id', 'path']])


,speaker_id,utt_id,label,system_id,path
0,T_0519,T_0000000994,bonafide,bonafide,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
1,T_0519,T_0000002182,spoof,A01,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
2,T_0519,T_0000003194,spoof,A02,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
3,T_0519,T_0000007618,spoof,A03,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...
4,T_0519,T_0000002372,spoof,A04,/home/SpeakerRec/BioVoice/data/datasets/ASVspo...


## Listen
Each audio widget below should correspond to the same speaker under a different system.


In [6]:
for _, row in listen_df.iterrows():
    wav_path = Path(row['path'])
    print(f"system={row['system_id']} | label={row['label']} | utt_id={row['utt_id']}")
    if wav_path.exists():
        display(Audio(filename=str(wav_path)))
    else:
        print('Missing file:', wav_path)


system=bonafide | label=bonafide | utt_id=T_0000000994


system=A01 | label=spoof | utt_id=T_0000002182


system=A02 | label=spoof | utt_id=T_0000003194


system=A03 | label=spoof | utt_id=T_0000007618


system=A04 | label=spoof | utt_id=T_0000002372
